In the previous post, we deconstructed a [payment transaction](deconstructing-a-payment-transaction/). Lets build on that, and look at verifying a transaction signature. 

The transaction has 11 inputs, and below are the details for the first input we are going to analyze.
```text 
Input 1 of 11 (bytes 5 to 184)
  Previous TXID : 3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404
  Output index  : 0  (bytes: 00 00 00 00)
  Script length : 139 bytes  (encoded as: 8b)
  Script bytes  :
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
  Sequence      : 4294967295  (bytes: ff ff ff ff)
```

There is no simple query to find which block a transaction exists on in the blockchain! There is no direct lookup that a node can perform. So, Bitcoin Core creates two databases, a UTXO DB (outputs that have not been spent), and an optional transaction index DB. Because I'm running a pruned node, I don't have the transaction index DB. So, i've used a block explorer to find the block hash for the above transaction.

Lets begin by getting the raw block of the block which holds the previous transaction using Bitcoin RPC API.

In [2]:
# $ bitcoin-cli getblockfrompeer 00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0 0
#   {}

# $ bitcoin-cli getblock 00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0 1
#   {
  "hash": "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0",
  "confirmations": 719521,
  "height": 144917,
  "version": 1,
  "versionHex": "00000001",
  "merkleroot": "da071791fd48d16c1ad9c1ebf50476f5e0271972f7424c9e627703e7cf2de26f",
  "time": 1315755730,
  "mediantime": 1315754014,
  "nonce": 1083621871,
  "bits": "1a096fe3",
  "difficulty": 1777774.482001465,
  "chainwork": "0000000000000000000000000000000000000000000000060725211546a9ab74",
  "nTx": 65,
  "previousblockhash": "00000000000005c6aca89f188819553e63fe1d5deeea9722b43d1cd0974c7d58",
  "nextblockhash": "00000000000008ff1b949dc5b9af89be744199e09073a58a78df9f7ce8355022",
  "strippedsize": 39882,
  "size": 39882,
  "weight": 159528,
  "tx": [
    "d2ffd4ea62db744fb1b98f6ab0127614c0dbb5b3eae86164876b9c63d6d47aa5",
    "3d5cae7fce60bb40e9e69d81fa8d356668e940c2127b8f5308be8bac5cd9babc",
    "659a046b5bc49e40e8a548704b1d68cd7657e327b8579997b1ff3f5d4acf553f",
    "2d677e0d38310730a31dda6a191d261440c38d69d8f53e806aa575c58155ab5a",
    "0ff84456947afc2c5bbe2924337fc031e3066975c481793548ff88e67feb4189",
    "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
    "1572a8bd96919f6bda2ce2f31f2fa089f04899cea54d617cd4b438f24ce0cab2",
    "33f4ba2b91785e2c9a7bd1a3c8d182274d3e2f4e896f3b2405fc683e6ec4d64d",
    "4f5ef2686fb343efae938766beb5c506b87646bdaa80b1516617b566d6bffa55",
    "219f53eb8ea18fbc56375defbd75573fb89c6bffe8844fc7a3f4dc0c59d2ca37",
    "c1a0505bcdd2e1a65cf2ba006fb573c8d1557c96fe91a323f53d0c340269893a",
    "9efab17469b5dbc4ff48c36453e4e19c422665d309fb20bae4d5e58aefd75e5a",
    "b68104a07f212f54275eff2b7e472b7bcc9fc694bc29d7cdde1412cd16c5a530",
    "dc854113ed1984bf5d8f4c7552035f294adea89accfe5cebcb02f80cdc5ec835",
    "215a7c3775514dc78af7a1385eee10cc5fe510630694df6f067254abf838cba3",
    "9783b46fe061ebcfc563c0d142beddeed7242518a272501619d2608c5c0e4ea4",
    "eb4db23872bf524071b30bd65a0707553f6c1027d6d784705b76c79541bd82ad",
    "644f4e7b0b71c6ee4acd6cf4e8dc2344602a01d5928926e3ed1475d63c71c5b6",
    "9690cdd47bbb23351a26ed6dc15b184b15c5f8d1f5603d955ed963aa95624e16",
    "09d584bdc2078cca3e2a7a5817221fb7876c5653910f5dd875f40399d26db28b",
    "dc5db167783d684ddad2ca6ed8ea987482d3278c091354342349d936f3eab69b",
    "5ab83ac7b1827a1eec94fc0efbc613533daad98ba76dc7701c51e27742eea6fb",
    "d320b51c4fda7bb0820544da7ddfeec67458ec5a04145b1e69bf536609d6ed26",
    "fe60e3c3b1d4fb052cf4434162c5b8696e3c5a49091897b4934f32a6bd961790",
    "77921e1c11918219661b174f236f6221691ad5821ea90ed54138f0802d596957",
    "7a3ac08253ef097f7f568c9e66a7ea1004af058b8b9fe34b682275dc13c37e6c",
    "e90c73de690f7e58db8d72b9de3b336379bc359c2a4030401e56902976193800",
    "a02772e73455c09aeb817de605283a5d7f2f26deb277e38922f6cb12f2201152",
    "0e58a67ce6245fc81e27bd38128835ee6cf7edeec51f9f86bb847119ebef8bb5",
    "d15f2ff9fc97a68e25288dc5539bf02110042995fa4ee7329461fa989d27695d",
    "df249dacd1c6826d02a4a876ef87e9f58d088b0bdc62b175d1b4bbd04acdff0f",
    "0d4ada2e08ef06b0b24a9f74e6368c2ca706f435d4058e96661a241d2a0d080f",
    "1c7c3ef4c7a7de668526e081d2bcb39c6b21316825f16f3b54e7b10b19930619",
    "4a006170a883fb37ef2ec745155ee14f19e32c40f19eaa17593660e1929d85c9",
    "8ce1e19ded159425a5197789b643b405cc06c92dec69c06b231ee1ec54fd427b",
    "cf624679fa63bc8b19c91ba5d3f0ecff900a618002ca865a45730600905089c6",
    "a62d113001193dfce5ff53f13f831713fbf73019c4167d8f53ed9c0c82b0250f",
    "f639a376194d39112aa1a050edc17855a47de5815c06045aa586572210dcface",
    "5981365960b7fd3f62f022a7f6533374820ee7697d9a01d7e48561556439800e",
    "6a2a4c9d41281c2ea4d1cd0e18eabebef4ce0aea31eeb75c4edcebdf8ae2252d",
    "0e11f562f41478c60c4b22afb0ea96087a1150691f257c8186486399726c28f3",
    "2fd5b36844089bacacc562682101b6b9b88077238adfccedbfd180890b1e8daf",
    "d8240b595c3d6d3546d30bfd0f44c89b92b2fcaf6128f4e148d059618adad84f",
    "6572f8863b149a0d3e0499851dfcb65dd022642c482f35870844ce5fcb7c6c40",
    "22b38dd0164ac932cd1aadcac2be1d28ab2d68ef5073879ef3e980bbd2315c02",
    "bda11440654ee164e95ad16b1560bbdd93da2d221ef1b022fdc9ec1ed8e0dc19",
    "5c2ba1a9d176727802d491ee57bd0493ffe7d78f417b194adc109e02f46189eb",
    "88b9e819884794a20de922f02f4f069bcdd3d137ce4aff6c161488ded9ee1daa",
    "7823d67d2563a1326b2acd110bc3fa96aecdf4c855aeafd5fbe5b0e594cd895f",
    "613d59ea86f47c616a0887776ad3f6946e8175e2d15c7b4ef14158626b187106",
    "b9f38202f39e189d34ff754bf5d2b8958e8ab6dd536dbb87360d64248a3ac97c",
    "ed86e9e0f27bcc758536ccf07799cae3921b16424e782916744b274e1a4fb789",
    "3de42d56f10b42cedf13d0460e64255edd796c06b11f8899ebc9477811f8cf93",
    "0cb213c7eb6dd7a366f7a396b58de7777790470276e0bae6f1e7586662c45e3e",
    "78efc44f30d15fe993a77c5fca58f0a73a91bff622b221cbe87c7cae14a28b6f",
    "9fa8bda4863b7097369c01f7b6dc02ced5acb75f260ba728e95e2035586fb886",
    "04ba17158e19138fc0bde8a7ab32633d82a69e3da6576e71278a852b59c92276",
    "d5dbb736561d563fa4a9e626bbbe286d35285d0f17a7099248a6d9e89e5a78a3",
    "3aa5935dc41a0cc3e045e5026a2eeb5d14a084213c9c561c51af542f2ec81bf0",
    "89c0e24513a88c4039fd2bdae54c9ac2ee84bfd88180e643bb772e3a17372f01",
    "6d0e0f502d28757df10e48d03b3aad90d6a52a776338c6065899fc5dc6d68a57",
    "e20e2a0d9cf577d11f4b8804d06d1fa045feeaac565057c9f37398e875f428c0",
    "eaf9cc5b7fa8260a7905714c71af33c504ea6ca100e56ba6486a0fce55eb002e",
    "8edf132399776ee6d311faf2f2152f71a01af8dc445d8d2bb1f2df06e3f0dc3d",
    "f3b795e40dab3d5767161dc2ceebc36da7c3021198269a3ef6296385244c5506"
  ]
}

IndentationError: unexpected indent (3829531532.py, line 6)

## What I learnt

- The blockchain is a database, but it's not like a normal DB. Instead, it's more like a snapshot to verify that things existed in a state on a previous date. For example, there is no simple query to find which block a transaction exists on! There is no direct lookup that a node can perform. So, Bitcoin Core creates two databases, a UTXO DB (outputs that have not been spent), and an optional transaction index DB. Because I'm running a pruned node, I don't have the transaction index DB.
- Because of the pruned node, I have to request any transaction, blocks, etc. from peers who are running a full archival node. But I still can't request a transaction from a peer without a block hash.
- Everything in the blockchain is stored in bytes. The bitcoin-cli and API convert it into hex text while displaying it for me.

  <div class="byte-example">
  <div class="byte-example-row">
  <code>00000000000005c6aca89f188819553e63fe1d5deeea9722b43d1cd0974c7d58</code>
  <span class="byte-example-label">Block hash in hex</span>
  </div>
  <div class="byte-example-row">
  <code>01 00 00 00 f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00 4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98 23 d5 6c 4e e3 6f 09 1a 23 d7 3f e4</code>
  <span class="byte-example-label">Block header in hex</span>
  </div>
  </div>
- In a transaction, an input spends a previous output. That previous output held the funds, and it has some spending conditions, which the spender has to pass to spend its funds. The input doesn't contain the amount it spent. It only has a reference to the previous output, which has the amount it spent.
- When I downloaded Bitcoin Core, it also provided me with an API called JSON-RPC. Basically, I can send a JSON request to my local Bitcoin Core, and it runs it and will return a response.
- Each block has a block header (80 bytes), which consists of the version (4 bytes), previous block header hash (32 bytes), merkle root hash (32 bytes), time (4 bytes), nBits (4 bytes), and nonce (4 bytes).

  <div class="byte-example">
  <div class="byte-example-row">
  <code>01 00 00 00</code>
  <span class="byte-example-label">version (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>f6 05 25 43 53 a7 6d 40 6d 62 8a 86 45 f7 5a 4b f6 48 64 61 c8 1e 76 b7 60 02 00 00 00 00 00 00</code>
  <span class="byte-example-label">previous block header hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>4f c5 cd ab 3d 9d 02 e4 bc 62 0a b7 97 80 9c 88 95 d7 6e 5b 9a 54 8c a9 8d f2 86 00 ec 8c 5f 98</code>
  <span class="byte-example-label">merkle root hash (32 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d5 6c 4e</code>
  <span class="byte-example-label">time (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>e3 6f 09 1a</code>
  <span class="byte-example-label">nBits (4 bytes)</span>
  </div>
  <div class="byte-example-row">
  <code>23 d7 3f e4</code>
  <span class="byte-example-label">nonce (4 bytes)</span>
  </div>
  </div>
- After the 80-byte block header, the next bytes represent the transaction count, which tells us how many transactions are included in the block. But the transaction count uses CompactSize which is not fixed. Basically, if the first byte falls between 00-'fc', then it means read only the 81st byte. If the first byte is 'fd', then read the next 2 bytes (82nd and 83rd), for 'fe' read next 4 bytes, for 'ff' read next 8 bytes. Or if it's easier to think in integers.. 0-252 is first.. then 253, 254 and 255 are the markers.
- I’m not using decoderawtransaction from Bitcoin Core for this problem.. So, looking up a previous transaction in an older block isn't very straightforward. For eg, each outpoint in a transaction input has the previous transaction id. BUT, this transaction id is actually a hash of all the data of that previous transaction. It doesn’t exist inside the previous transaction. It's there in the blocks merkle root which allows us to verify that the transaction exists in the block. But as I want to find selected transactions to read the outputs from it, I have to hash each one of the blocks transactions, create the transaction hash first, and if chosen, get the transaction outputs. For each transaction I want, the block already has the exact bytes I need to hash. But I need to find the start and position for each transaction from the block.
- To find the start, end for each transaction.. I need to increment the position by the correct field size. Some of the transaction fields are or fixed bytes.. e.g. version, lock\_time, the previous\_output etc.. But many of them are of variable size of type compactSize. For e.g. the number of transactions in a block, the number of input transactions in a transaction, the public\_key script. So, the program needs to update the position by finding the compactSize for each of them. Doing all of this will give us the start and end position for each transaction.. Which we can then use to find the exact bytes for each transaction and then hash it to get the id.

